# V-cloak — GPU job runner

This notebook runs one V-cloak **GPU job** on a free cloud GPU and gives you back a results file.
You never edit code here. Five steps, about two minutes of your attention.

1. **Turn on the GPU.** Colab: `Runtime → Change runtime type → T4 GPU → Save`.
   Kaggle: `Settings → Accelerator → GPU T4 x2` (phone-verify your account once, if asked).
2. Run **Cell 1** to check the GPU is really there.
3. Run **Cell 2** and upload the `job.zip` that Claude packed for you
   (it is in your repo at `gpujobs/out/<job-id>/job.zip`).
4. Run **Cell 3**. This does the real work — it may take anywhere from one minute to about forty,
   depending on the job. Leave the tab open and visible; both services stop idle notebooks.
5. Run **Cell 4** to download `results_<job-id>.zip`. Put that file in your repo's
   `gpujobs/inbox/` folder, open a new Claude Code session, and type `continue`.

**If anything fails,** don't fix it here. Run Cell 5 and give Claude the printed summary — the job
is designed so a failure is useful evidence, not a dead end.

## Cell 1 — Is the GPU really attached?

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")
try:
    import cupy, numpy
    print("CuPy", cupy.__version__, "| NumPy", numpy.__version__)
    d = cupy.cuda.Device(0)
    free, total = cupy.cuda.runtime.memGetInfo()
    props = cupy.cuda.runtime.getDeviceProperties(0)
    print("Device:", props["name"].decode(), "| CC",
          f'{props["major"]}.{props["minor"]}', "| VRAM free/total GB:",
          round(free/1e9, 2), "/", round(total/1e9, 2))
    print("\nGPU is ready. Go to Cell 2.")
except ImportError:
    print("CuPy is missing. Installing (2-3 min) — then re-run this cell.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=False)

## Cell 2 — Upload the job bundle
Upload `job.zip` from `gpujobs/out/<job-id>/`. On Kaggle the file picker is not available: instead add the
zip as a notebook **Dataset** (`+ Add Input → Upload`), then set `KAGGLE_INPUT` below to its path.

In [ ]:
import os, zipfile, pathlib, shutil, json

WORK = pathlib.Path("/content/vcloak_job") if pathlib.Path("/content").exists() else pathlib.Path("/kaggle/working/vcloak_job")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

KAGGLE_INPUT = ""   # e.g. "/kaggle/input/vcloak-job/job.zip" — leave empty on Colab

if KAGGLE_INPUT:
    src = pathlib.Path(KAGGLE_INPUT)
else:
    from google.colab import files           # noqa: F401
    up = files.upload()
    src = pathlib.Path(list(up.keys())[0])

with zipfile.ZipFile(src) as z:
    z.extractall(WORK)
# the bundle may be nested one level deep
if not (WORK / "job.json").exists():
    inner = next((p for p in WORK.iterdir() if (p / "job.json").exists()), None)
    if inner:
        WORK = inner
job = json.loads((WORK / "job.json").read_text())
print("Job:", job["job_id"], "| module:", job.get("module"), "| quick:", job.get("quick"))
print("Commands:")
for i, c in enumerate(job.get("commands", []), 1):
    print(f"  [{i}] {c['cmd'] if isinstance(c, dict) else c}")

## Cell 3 — Run the job
Keep this tab open and in view. If the runtime disconnects, re-run Cells 1–3; the job restarts cleanly.

In [ ]:
import subprocess, sys, os
env = dict(os.environ)
env["VCLOAK_RUNNER"] = "kaggle" if os.path.exists("/kaggle") else "colab"
p = subprocess.run([sys.executable, "gpu_entry.py", "--job", "job.json"],
                   cwd=str(WORK), env=env, text=True)
print("\nrunner exit code:", p.returncode, "(0 = every command passed)")

## Cell 4 — Download the results

In [ ]:
import glob, pathlib
hits = sorted(glob.glob(str(WORK / "results_*.zip")))
print("found:", hits)
if hits:
    path = hits[-1]
    size = pathlib.Path(path).stat().st_size / 1e6
    print(f"{path}  ({size:.1f} MB)")
    try:
        from google.colab import files
        files.download(path)
        print("Downloading... then move it into your repo at gpujobs/inbox/ and type: continue")
    except ImportError:
        print("Kaggle: find this file in the Output panel on the right and download it from there.")
else:
    print("No results file. Run Cell 5 and send Claude the summary.")

## Cell 5 — Only if something went wrong

In [ ]:
import json, pathlib
rp = WORK / "result.json"
if rp.exists():
    r = json.loads(rp.read_text())
    print("job:", r["job_id"], "| overall exit:", r["overall_exit"])
    print("env:", json.dumps({k: r["env"].get(k) for k in ("gpu","driver","cupy","nvcc","ncu","runner")}, indent=1))
    for c in r["commands"]:
        print(f"  [{c['n']}] {c['name']}: exit {c['exit_code']}")
    bad = [c for c in r["commands"] if c["exit_code"] != 0]
    for c in bad[:2]:
        print("\n--- tail of", c["name"], "---")
        print("\n".join((WORK / c["log"]).read_text().splitlines()[-40:]))
else:
    print("result.json missing — the job did not start. Re-run Cell 1 and check the GPU is attached.")